# Catch Radius Pressure (CRP) — NFL Big Data Bowl 2026
### Colab-Ready Analysis Notebook

This notebook works out-of-the-box in **Google Colab**. It will:
1. Clone the repo (or use existing files)
2. Auto-detect the data location
3. Run the full CRP analysis

---

## Overview

Traditional completion metrics treat all catches as equal. A receiver catching a ball wide open and a receiver making a contested grab through traffic are rated identically.

**Catch Radius Pressure (CRP)** quantifies how much defensive pressure exists at the exact location the ball arrives — combining defender proximity and velocity into a single, interpretable score.

> **Higher CRP = More contested, more difficult catch situation**


---
## 0. Setup (Colab)

Run this cell first. It clones the repo if needed and sets up imports.


In [ ]:
# ── Colab setup ──────────────────────────────────────────────────────────
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
REPO_NAME = 'catch-radius-pressure'

if IN_COLAB:
    # Clone the repo if not already there
    if not os.path.isdir(f'/content/{REPO_NAME}'):
        # ⚠️ Replace with your actual GitHub URL after you push:
        # subprocess.run(['git', 'clone', 'https://github.com/YOUR_USERNAME/catch-radius-pressure'], cwd='/content')
        print("Repo not detected at /content/catch-radius-pressure")
        print("Either:")
        print("  (a) Upload the zip to Colab and unzip with:")
        print("      !unzip catch_radius_pressure_project.zip -d /content/")
        print("  (b) After pushing to GitHub, replace the URL above and uncomment the git clone line")
    REPO_PATH = f'/content/{REPO_NAME}'
else:
    # Local: assume notebook is in /notebooks/ or repo root
    here = os.path.abspath('.')
    if os.path.basename(here) == 'notebooks':
        REPO_PATH = os.path.dirname(here)
    elif os.path.isdir(os.path.join(here, 'crp')):
        REPO_PATH = here
    else:
        REPO_PATH = here  # fallback

sys.path.insert(0, REPO_PATH)
os.chdir(REPO_PATH)
print(f"Repo path: {REPO_PATH}")
print(f"Contents: {sorted(os.listdir(REPO_PATH))[:10]}")


In [ ]:
# ── Install dependencies (Colab usually has these, but just in case) ─────
if IN_COLAB:
    !pip install -q pandas numpy matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from crp.metric import compute_crp_for_play, add_crp_labels, CATCH_RADIUS_YDS
from crp.data_loader import get_play_snapshot
from crp.visualizations import (
    plot_crp_distribution, plot_crp_vs_completion,
    plot_crp_heatmap, plot_crp_by_coverage, plot_play_snapshot
)

plt.rcParams['figure.facecolor'] = '#111111'
plt.rcParams['axes.facecolor'] = '#1a1a1a'
plt.rcParams['text.color'] = 'white'
print("✓ Imports OK")


---
## 1. Load Pre-Computed CRP Data

CRP has been pre-computed across all 14,108 passing plays from the 2023 NFL season.  
The `crp_merged.csv` file (CRP + supplementary play metadata) is included in the repo.


In [ ]:
df = pd.read_csv('data/crp_merged.csv')
print(f"Plays loaded: {len(df):,}")
df[['game_id', 'play_id', 'crp', 'crp_label', 'n_defenders_in_radius',
    'pass_result', 'team_coverage_type', 'route_of_targeted_receiver', 'yards_gained']].head(10)


---
## 2. CRP Formula

For each pass play, CRP is computed at ball arrival:

$$\text{CRP} = \sum_{i \in D_R} \left(1 - \frac{d_i}{R}\right) \cdot (1 + v_i)$$

Where:
- $D_R$ = set of defenders within catch radius $R$ (default: **3 yards**)
- $d_i$ = distance (yards) from defender $i$ to ball landing spot
- $v_i$ = velocity component (yards/frame) of defender $i$ **toward** the ball (floored at 0)
- $R$ = catch radius in yards (3.0)

**Interpretation:**
- A defender at the exact catch point moving directly toward it at speed contributes ~1.0+
- A defender at the edge of the radius barely moving contributes near 0
- Multiple defenders stack — Extreme Pressure plays can exceed 2.0


---
## 3. CRP Distribution Across the Season


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_crp_distribution(df, ax=ax)
plt.tight_layout()
plt.show()

print("\nCRP Statistics:")
print(df['crp'].describe().round(4))
print("\nLabel Distribution:")
print(df['crp_label'].value_counts())


**Key observations:**
- ~57% of passing plays are Open (CRP = 0) — no defender within the 3-yard catch radius at arrival
- The distribution is right-skewed; truly contested catches are relatively rare
- Max CRP observed: **2.56** — three defenders converging inside the catch radius simultaneously


---
## 4. CRP and Completion Rate

Does higher pressure actually reduce completion probability?


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_crp_vs_completion(df, ax=ax)
plt.tight_layout()
plt.show()

comp = df[df['pass_result'].isin(['C', 'I'])].copy()
comp['completed'] = (comp['pass_result'] == 'C').astype(int)
summary = comp.groupby('crp_label')['completed'].agg(['mean', 'count'])
summary.columns = ['Completion Rate', 'Play Count']
summary['Completion Rate'] = (summary['Completion Rate'] * 100).round(1).astype(str) + '%'
print(summary.sort_values('Completion Rate', ascending=False).to_string())


**Key finding:** Completion rate drops monotonically with CRP pressure level:
- **Open** (CRP = 0): **79.6%** completion rate
- **High Pressure** (CRP 1.0–1.5): **45.6%** completion rate
- **Extreme Pressure** (CRP > 1.5): **37.5%** completion rate

This validates that CRP is meaningfully correlated with play difficulty — not just a theoretical construct.


---
## 5. Where on the Field Does Pressure Concentrate?


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
plot_crp_heatmap(df, ax=ax)
plt.tight_layout()
plt.show()


**Observations:**
- Pressure concentrates near the sidelines and in the red zone (final 20 yards)
- Middle-of-field routes (10–20 yards downfield) tend to be more open
- Boundary throws attract more defensive attention, especially on GO and CORNER routes


---
## 6. CRP by Coverage Scheme


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
plot_crp_by_coverage(df, ax=ax)
plt.tight_layout()
plt.show()

print("\nMean CRP by Coverage:")
print(df.groupby('team_coverage_type')['crp'].mean().sort_values(ascending=False).round(4).to_string())


**Man coverage generates more catch-point pressure than zone:**
- COVER_1_MAN and COVER_2_MAN produce the highest average CRP (0.30+)
- Zone coverages like COVER_2_ZONE (0.15) create less concentrated pressure at the catch point
- This aligns with football intuition: man defenders follow receivers to the ball; zone players cover areas and may not converge as tightly


---
## 7. CRP by Route Type


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#111111')

route_crp = df.groupby('route_of_targeted_receiver')['crp'].mean().sort_values(ascending=False).dropna()
axes[0].set_facecolor('#1a1a1a')
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(route_crp)))
axes[0].barh(route_crp.index, route_crp.values, color=colors)
axes[0].set_xlabel('Avg CRP', color='white')
axes[0].set_title('Average CRP by Route', color='white')
axes[0].tick_params(colors='white')
axes[0].spines[:].set_color('#444444')

route_stats = df[df['pass_result'].isin(['C','I'])].copy()
route_stats['completed'] = (route_stats['pass_result']=='C').astype(int)
rs = route_stats.groupby('route_of_targeted_receiver').agg(
    comp_rate=('completed','mean'), avg_crp=('crp','mean'), n=('completed','count')
).reset_index().dropna()
sc = axes[1].scatter(rs['avg_crp'], rs['comp_rate']*100,
    c=rs['avg_crp'], cmap='RdYlGn_r', s=rs['n']/5, alpha=0.85, edgecolors='white', linewidths=0.8)
for _, row in rs.iterrows():
    axes[1].annotate(row['route_of_targeted_receiver'],
        (row['avg_crp']+0.003, row['comp_rate']*100), fontsize=8, color='white', alpha=0.9)
axes[1].set_xlabel('Avg CRP', color='white')
axes[1].set_ylabel('Completion Rate (%)', color='white')
axes[1].set_title('Route: CRP vs Completion Rate', color='white')
axes[1].set_facecolor('#1a1a1a')
axes[1].tick_params(colors='white')
axes[1].spines[:].set_color('#444444')

plt.tight_layout()
plt.show()


**GO routes face the most pressure (avg CRP 0.477)** — defenders play tight coverage on deep shots.  
**HITCH and OUT routes** are the safest (lowest CRP), explaining their use in West Coast / quick-game offenses as completion percentage boosters.


---
## 8. Play-Level Examples

### High CRP Play — Defenders Converging on the Catch Point

**Note:** This section requires the raw competition data in `data/114239_nfl_competition_files_published_analytics_final/`.  
If you don't have it locally, skip this section or upload the data files first.


In [ ]:
DATA = 'data/114239_nfl_competition_files_published_analytics_final'

if not os.path.isdir(DATA):
    print(f"⚠️ Raw data not found at {DATA}")
    print("Skipping play snapshots. Pre-computed stats above still work.")
    HAS_RAW = False
else:
    HAS_RAW = True
    top = df[(df['pass_result']=='C')].sort_values('crp', ascending=False).iloc[0]
    print(f"Game: {top['game_id']}  Play: {top['play_id']}")
    print(f"CRP: {top['crp']:.3f} | Defenders in radius: {top['n_defenders_in_radius']}")
    print(f"Route: {top.get('route_of_targeted_receiver', 'N/A')} | Coverage: {top.get('team_coverage_type', 'N/A')}")
    print(f"Yards gained: {top.get('yards_gained', 'N/A')}")
    print(f"Play: {top.get('play_description', 'N/A')}")


In [ ]:
if HAS_RAW:
    week = int(top['week_x'])
    df_w = pd.read_csv(f'{DATA}/train/input_2023_w{week:02d}.csv', low_memory=False)
    snap = get_play_snapshot(df_w, int(top['game_id']), int(top['play_id']), frame='last')

    fig, ax = plt.subplots(figsize=(14, 6))
    fig.patch.set_facecolor('#111111')
    plot_play_snapshot(snap, top['ball_land_x'], top['ball_land_y'],
        title=f'High CRP Completion | CRP = {top["crp"]:.3f} | Route: {top.get("route_of_targeted_receiver","")}',
        ax=ax)
    plt.tight_layout()
    plt.show()


### Low CRP Play — Wide Open Catch

In [ ]:
if HAS_RAW:
    low = df[(df['pass_result']=='C') & (df['crp']>0)].sort_values('crp').iloc[0]
    week2 = int(low['week_x'])
    df_w2 = pd.read_csv(f'{DATA}/train/input_2023_w{week2:02d}.csv', low_memory=False)
    snap2 = get_play_snapshot(df_w2, int(low['game_id']), int(low['play_id']), frame='last')

    fig, ax = plt.subplots(figsize=(14, 6))
    fig.patch.set_facecolor('#111111')
    plot_play_snapshot(snap2, low['ball_land_x'], low['ball_land_y'],
        title=f'Low CRP Completion | CRP = {low["crp"]:.3f} | Route: {low.get("route_of_targeted_receiver","")}',
        ax=ax)
    plt.tight_layout()
    plt.show()


---
## 9. Summary & Conclusions

**Catch Radius Pressure (CRP)** provides a new lens for evaluating passing plays:

| Finding | Detail |
|---------|--------|
| **CRP predicts completion difficulty** | Completion rate drops from 79.6% (Open) to 37.5% (Extreme Pressure) |
| **Man coverage generates more pressure** | Man schemes average 0.30+ CRP vs 0.15–0.21 for zone |
| **GO and POST routes face the most pressure** | Deep shots (GO: 0.477 avg CRP) draw tight coverage |
| **Red zone and sideline = highest pressure zones** | Defenders concentrate at boundary catch points |
| **High-CRP completions are rare but impactful** | These contested catches often represent big gains |

### Future Directions
- **Receiver grades**: rank receivers by catch rate over expected given their CRP exposure
- **Quarterback bravery index**: do QBs throw into pressure or avoid contested areas?
- **Coverage scheme efficiency**: which defenses generate the most CRP per play?
- **Temporal CRP**: track how pressure builds frame-by-frame during ball flight

---
*NFL Big Data Bowl 2026 | Catch Radius Pressure Project*
